# 第二轮困难正样本消融与蒸馏

固定原 Pilot 的 470/75/50 互斥划分；先训练三个监督模型，再提取 Teacher 并训练三个 KD 变体。模型和阈值仅用验证集选择，最终只对入选 Baseline 与 KD 各评估一次固定测试集。固定测试集已在上一轮被查看，因此不再视为盲测。

In [ ]:
import json
import os
import subprocess
import sys
import time
from datetime import datetime, timezone
from pathlib import Path
import torch

if not torch.cuda.is_available():
    raise RuntimeError('未分配Kaggle GPU')
print('GPU:', torch.cuda.get_device_name(0))
os.environ['PYTORCH_CUDA_ALLOC_CONF'] = 'expandable_segments:True'


In [ ]:
repo_dir = Path('/kaggle/working/internvl-pv-distillation')
repo_url = 'https://github.com/Zh0416/internvl-pv-distillation.git'
fixed_commit = '524ac7386f189e62b81de38de442c53782b8f5ea'
output_dir = Path('/kaggle/working/round2_outputs')
stage_log_dir = output_dir / 'stage_logs'
stage_log_dir.mkdir(parents=True, exist_ok=True)
stage_status_path = output_dir / 'reports/round2_execution_status.json'
stage_status_path.parent.mkdir(parents=True, exist_ok=True)
stage_records = []

def run_external(stage, command, timeout_seconds, cwd=None):
    record = {'stage': stage, 'status': 'running', 'started_at': datetime.now(timezone.utc).isoformat(), 'timeout_seconds': timeout_seconds}
    stage_records.append(record)
    stage_status_path.write_text(json.dumps(stage_records, ensure_ascii=False, indent=2), encoding='utf-8')
    log_path = stage_log_dir / f'{stage}.log'
    print(f'STAGE START: {stage} (timeout={timeout_seconds}s)', flush=True)
    started = time.perf_counter()
    try:
        with log_path.open('w', encoding='utf-8') as log_file:
            subprocess.run(list(map(str, command)), cwd=cwd, stdout=log_file, stderr=subprocess.STDOUT, timeout=timeout_seconds, check=True)
    except Exception as error:
        record['status'] = 'timeout' if isinstance(error, subprocess.TimeoutExpired) else 'error'
        record['error_type'] = type(error).__name__
        record['exit_code'] = getattr(error, 'returncode', None)
        raise
    else:
        record['status'] = 'complete'
    finally:
        record['finished_at'] = datetime.now(timezone.utc).isoformat()
        record['runtime_seconds'] = time.perf_counter() - started
        record['log_path'] = str(log_path)
        stage_status_path.write_text(json.dumps(stage_records, ensure_ascii=False, indent=2), encoding='utf-8')
        print(f"STAGE END: {stage} status={record['status']} runtime={record['runtime_seconds']:.1f}s", flush=True)

run_external('github_clone', ['git', 'clone', repo_url, str(repo_dir)], 300)
run_external('github_checkout', ['git', '-C', str(repo_dir), 'checkout', '--detach', fixed_commit], 60)
actual_commit = subprocess.check_output(['git', '-C', str(repo_dir), 'rev-parse', 'HEAD'], text=True).strip()
if actual_commit != fixed_commit:
    raise RuntimeError(f'Git commit不一致: {actual_commit}')
print('GitHub commit:', actual_commit)
run_external('install_requirements', [sys.executable, '-m', 'pip', 'install', '--quiet', '-r', str(repo_dir / 'requirements.txt')], 900)
config_path = repo_dir / 'configs/kaggle_pv_pilot_distill.yaml'

def run_script(name, *arguments, stage=None, timeout_seconds=1200):
    run_external(stage or Path(name).stem, [sys.executable, '-u', str(repo_dir / 'scripts' / name), *arguments], timeout_seconds, cwd=repo_dir)

def run_student(variant, run_name, extra=(), evaluate_only=False, thresholds=None, test=False):
    command = ['--config', config_path, '--output-dir', output_dir, '--variant', variant, '--run-name', run_name]
    if evaluate_only:
        command.append('--evaluate-only')
        command.extend(['--thresholds', thresholds])
        if test:
            command.append('--test')
    else:
        command.extend(['--epochs', '12', '--defer-test', *extra])
    stage = f"{run_name}_{'test' if test else 'scan' if evaluate_only else 'train'}"
    run_script('10_train_pilot_student.py', *command, stage=stage, timeout_seconds=1200)


In [ ]:
run_script('08_prepare_pilot_manifest.py', '--config', config_path, '--output-dir', output_dir, stage='prepare_manifest')
manifest = json.loads((output_dir / 'reports/pilot_manifest.json').read_text(encoding='utf-8'))
keys = [row['cache_key'] for rows in manifest['splits'].values() for row in rows]
if len(keys) != 595 or len(set(keys)) != 595:
    raise RuntimeError('划分数量或唯一性不符')
print(json.dumps(manifest['counts'], ensure_ascii=False, indent=2))


In [ ]:
baseline_runs = {
    'reference': (),
    'hard_weight': ('--hard-positive-weight', '2.5'),
    'positive_weight': ('--positive-pixel-weight', '2.0'),
}
for run_name, extra in baseline_runs.items():
    run_student('baseline', run_name, extra)
    run_student('baseline', run_name, evaluate_only=True, thresholds='0.3,0.4,0.5,0.6')
run_script('12_select_round2.py', '--output-dir', output_dir, '--stage', 'baseline', stage='select_baseline')
selection_path = output_dir / 'reports/round2_selection.json'
selection = json.loads(selection_path.read_text(encoding='utf-8'))
baseline = selection['baseline']
print('Selected baseline:', baseline['run_name'], 'threshold:', baseline['threshold'])


In [ ]:
run_script('09_extract_pilot_teacher.py', '--config', config_path, '--output-dir', output_dir, stage='extract_teacher', timeout_seconds=7200)


In [ ]:
baseline_parameters = baseline['parameters']
supervision = ['--positive-pixel-weight', str(baseline_parameters['positive_pixel_weight'])]
if baseline_parameters['hard_positive_weight'] is not None:
    supervision.extend(['--hard-positive-weight', str(baseline_parameters['hard_positive_weight'])])
kd_runs = {
    'semantic_only': ('--semantic-kd-weight', '0.1', '--feature-kd-weight', '0.0'),
    'feature_only': ('--semantic-kd-weight', '0.0', '--feature-kd-weight', '0.05'),
    'combined_low': ('--semantic-kd-weight', '0.1', '--feature-kd-weight', '0.05'),
}
for run_name, kd_parameters in kd_runs.items():
    run_student('kd', run_name, (*supervision, *kd_parameters, '--feature-agreement-only'))
    run_student('kd', run_name, evaluate_only=True, thresholds='0.3,0.4,0.5,0.6')
run_script('12_select_round2.py', '--output-dir', output_dir, '--stage', 'kd', stage='select_kd')
selection = json.loads(selection_path.read_text(encoding='utf-8'))
print('Selected KD:', selection['kd']['run_name'], 'validation guard:', selection['kd_validation_guard_passed'])


In [ ]:
for variant, selected in (('baseline', selection['baseline']), ('kd', selection['kd'])):
    run_student(variant, selected['run_name'], evaluate_only=True, thresholds=str(selected['threshold']), test=True)
run_script('12_select_round2.py', '--output-dir', output_dir, '--stage', 'finalize', stage='finalize')
selection = json.loads(selection_path.read_text(encoding='utf-8'))
print('Accepted:', selection['test_comparison']['accepted'])
print('Test deltas:', json.dumps(selection['test_comparison']['delta'], ensure_ascii=False, indent=2))
